# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drie01/flyranks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (connect to the warehouse)

Same recipe as notebook 03: the token comes from an env var, a Colab Secret named `HF_TOKEN`, or a prompt — **never pasted into a cell** (this repo is public).

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

In [ ]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Lane: Ranking Signal Analysis** — *which safe search signals are associated with later clicks?*

**Contract answers 1–3 (plain words):**

1. **What one row means.** In the warehouse table, one row is one content item on one day (`report_date × client × content`). In my feature frame I roll those days up, so one row is **one content item (client × content) in March 2026**.
2. **Which table.** Only `fact_content_daily_performance`, the `month=2026-03` partition. Not the `_sample` table (that is June 2026, the sealed final month), and no other tables yet.
3. **Which time window.** The decision moment is the **end of 15 March 2026**. Features use days **1–15 March**. The outcome I rank by lives in days **16–31 March**. Nothing from the outcome window may be used as a feature.

In [ ]:
# The contract as constants, so every query below uses exactly the same windows.
REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

M_START   = '2026-03-01'   # first day of the month
EARLY_END = '2026-03-15'   # last day a feature may use (the decision moment)
LATE_START = '2026-03-16'  # first day of the outcome window
M_END     = '2026-04-01'   # exclusive

# Setup check (not one of the three verification queries): do the columns I rely on exist?
cols = con.sql(f"SELECT * FROM {FACT} LIMIT 0").columns
need = ['report_date', 'client_hash_id', 'content_hash_id',
        'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_data_available']
missing = [c for c in need if c not in cols]
assert not missing, f'columns not found: {missing}. Actual columns: {cols}'
print('all columns I use are present:', need)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Contract answers 4–5 (plain words):**

4. **What I rank / predict (a proxy, not a product decision).** Items ranked by **clicks in 16–31 March** (`late_clicks`). For a quick score I turn that into a yes/no label: *is this item in the top 10% of late clicks?* The 10% cut-off is learned on training clients only. This is an observed future outcome, not a rule someone wrote.
5. **One thing I deliberately exclude.** Every daily value dated **16–31 March**. It is the outcome window, so it may only ever be used to build the label, never a feature.

| Bucket | Fields |
|---|---|
| Feature (all from 1–15 Mar) | `early_impressions`, `early_clicks`, `early_avg_position`, `early_days_with_impr`, `early_pos_std` |
| Label / proxy | `late_clicks` (sum of `gsc_clicks`, 16–31 Mar) → `high_late_clicks` |
| Context (group / split only) | `client_hash_id`, `content_hash_id` — pseudonyms, never features |
| Excluded | anything dated 16–31 Mar except as the label source; `_sample` table (June 2026, sealed test month); `ga4_data_available` and GA4 metrics (checked in query 3, not used as features) |

In [ ]:
FEATURES = ['log_early_impressions', 'log_early_clicks', 'early_avg_position',
            'early_days_with_impr', 'early_pos_std']
LABEL_SOURCE = 'late_clicks'          # label built from this, never a feature
CONTEXT = ['client_hash_id', 'content_hash_id']
EXCLUDED = {'days 16-31 Mar (except as label source)': 'outcome window',
            '_sample table / June 2026': 'sealed final month',
            'GA4 columns': 'usable only where ga4_data_available IS TRUE (see query 3)'}
assert LABEL_SOURCE not in FEATURES and not set(CONTEXT) & set(FEATURES)
print('features:', FEATURES)
print('label source:', LABEL_SOURCE)

## 3. Verify it with queries (grain, counts, availability) — then features and the trap

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

All three queries run on the mid-panel month `2026-03`.

### Query 1 — the grain
Claim: one row is one `report_date × client × content`. Zero rows back means the grain holds.

In [ ]:
dups = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print('rows that break the grain (want 0):', len(dups))
dups

### Query 2 — row count and date span of my slice
Claim: the slice is the March 2026 partition only — dates 2026-03-01 to 2026-03-31.

In [ ]:
span = con.sql(f"""
    SELECT COUNT(*)                        AS n_rows,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT report_date)     AS n_days,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {FACT}
""").df()
assert str(span.loc[0, 'first_day'])[:7] == MONTH and str(span.loc[0, 'last_day'])[:7] == MONTH
span

### Query 3 — availability (`IS TRUE`, not `= TRUE`)
Claim: the GA4 flag is **three-valued** (TRUE / FALSE / NULL). `= FALSE` or `NOT ...` silently drops the NULL rows, so I filter with `IS TRUE` and count each state separately.

In [ ]:
avail = con.sql(f"""
    SELECT COUNT(*)                                            AS n_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_false,
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_null
    FROM {FACT}
""").df()
assert avail.loc[0, ['ga4_true', 'ga4_false', 'ga4_null']].sum() == avail.loc[0, 'n_rows']
avail['share_surviving_IS_TRUE'] = (avail['ga4_true'] / avail['n_rows']).round(4)
avail

### Five features — each one is knowable at the decision moment (end of 15 March)

Features are built from days **1–15 March** only (not a verification query — this is the feature build).

| Feature | Available when? |
|---|---|
| `log_early_impressions` | Knowable at the decision moment because it only adds up search impressions from days 1–15 March, all of which have already happened. |
| `log_early_clicks` | Knowable at the decision moment because it only adds up clicks from days 1–15 March. |
| `early_avg_position` | Knowable at the decision moment because it averages the daily search position on days 1–15 March (a `0` position means "no data", so it is left out of the average). |
| `early_days_with_impr` | Knowable at the decision moment because it counts days in 1–15 March with at least one impression — a calendar fact about the past. |
| `early_pos_std` | Knowable at the decision moment because it measures how much the daily position moved within 1–15 March. |

In [ ]:
import numpy as np, pandas as pd

df = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '{EARLY_END}')              AS early_impressions,
           SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '{EARLY_END}')              AS early_clicks,
           AVG(NULLIF(gsc_avg_position, 0)) FILTER (WHERE report_date <= DATE '{EARLY_END}')  AS early_avg_position,
           COUNT(*) FILTER (WHERE report_date <= DATE '{EARLY_END}' AND gsc_impressions > 0)  AS early_days_with_impr,
           STDDEV_SAMP(NULLIF(gsc_avg_position, 0)) FILTER (WHERE report_date <= DATE '{EARLY_END}') AS early_pos_std,
           COALESCE(SUM(gsc_clicks) FILTER (WHERE report_date >= DATE '{LATE_START}'), 0)     AS late_clicks
    FROM {FACT}
    WHERE report_date >= DATE '{M_START}' AND report_date < DATE '{M_END}'
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '{EARLY_END}') > 0
""").df()

df['early_clicks'] = df['early_clicks'].fillna(0)
df['log_early_impressions'] = np.log1p(df['early_impressions'])
df['log_early_clicks'] = np.log1p(df['early_clicks'])
print(f'{len(df):,} content items in the feature frame')
df[['client_hash_id', 'content_hash_id'] + FEATURES + [LABEL_SOURCE]].head()

### The honest number first

Split **by client** (so the test clients are never seen in training), learn the 10% cut-off on training clients only, then score a quick model with ROC-AUC. A one-column baseline (`early_clicks` alone) gives the number to beat.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(df, groups=df['client_hash_id']))
train, test = df.iloc[tr_idx].copy(), df.iloc[te_idx].copy()

thr = max(1, train[LABEL_SOURCE].quantile(0.90))     # learned on training clients only
for d in (train, test):
    d['high_late_clicks'] = (d[LABEL_SOURCE] >= thr).astype(int)
print(f'cut-off: >= {thr:g} late clicks | base rate train {train.high_late_clicks.mean():.3f}, test {test.high_late_clicks.mean():.3f}')

def quick_auc(cols):
    m = HistGradientBoostingClassifier(max_iter=100, random_state=42).fit(train[cols], train['high_late_clicks'])
    return roc_auc_score(test['high_late_clicks'], m.predict_proba(test[cols])[:, 1])

baseline_auc = roc_auc_score(test['high_late_clicks'], test['early_clicks'])
honest_auc = quick_auc(FEATURES)
print(f'one-column baseline (early_clicks): AUC {baseline_auc:.3f}')
print(f'honest 5-feature model:             AUC {honest_auc:.3f}')

### The trap — add ONE label-derived column on purpose

I add `leak_late_clicks_log`, a transformed copy of the outcome I am ranking by. It looks like "just another feature", but it is built from days 16–31 March — **after** the decision moment. Watch the score jump toward perfect, then delete the column and keep the honest number.

In [ ]:
# 1) add the leak
for d in (train, test):
    d['leak_late_clicks_log'] = np.log1p(d[LABEL_SOURCE])
leak_auc = quick_auc(FEATURES + ['leak_late_clicks_log'])
print(f'WITH the leaked column:    AUC {leak_auc:.3f}   <- looks perfect, means nothing')

# 2) delete the leak
train = train.drop(columns='leak_late_clicks_log')
test = test.drop(columns='leak_late_clicks_log')
assert 'leak_late_clicks_log' not in train.columns and 'leak_late_clicks_log' not in test.columns

# 3) keep the honest number
honest_after = quick_auc(FEATURES)
assert leak_auc > honest_auc, 'the leak should inflate the score'
assert abs(honest_after - honest_auc) < 1e-9, 'honest score must be unchanged once the leak is gone'
print(f'AFTER deleting the leak:   AUC {honest_after:.3f}   <- the number I keep')

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation of my slice: it is search-only.** My five features come from search (GSC) columns. The GA4 columns are usable only on rows where `ga4_data_available IS TRUE`, and query 3 shows what share of March rows that is — everything else is zero-filled (FALSE) or empty (NULL), not "no engagement". So this slice can say nothing about engagement signals, and any claim stays **observed and directional**: I watched which early search signals go with later clicks, I did not test what causes them.

In [ ]:
# Everything the notebook claims, in one place.
print('rows in feature frame :', f'{len(df):,}')
print('GA4 rows surviving IS TRUE (query 3):', f"{avail.loc[0, 'share_surviving_IS_TRUE']:.2%}")
print('honest AUC kept       :', f'{honest_after:.3f}')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.